# URLSentra — Member 4: Model Evaluation + SVM
**Owner:** Dharmasiri S.O.R. · **Module:** EC5203 Machine Learning · **Proposal:** sections 4.9, 4.10 and 4.12

**Status:** A reproducible training-only evaluation notebook. Recorded metrics and figures are generated by executing cells against the actual PhiUSIIL dataset, not invented. This notebook is intended to work **before** Member 3's final `features.py` merge, but the engineered-feature results must be rerun afterward.

**Protocol:** Target `1 = phishing`; model selection uses shared five-fold stratified CV on training rows. Test data are never evaluated during ordinary **Run All**. `URLSimilarityIndex` is excluded under the group's November 19 decision.

## 1. Settings and imports
Place `PhiUSIIL_Phishing_URL_Dataset.csv` under `data/raw/` (download: https://archive.ics.uci.edu/dataset/967). Run the notebook from the repository's `notebooks/` directory or the project root. `SAMPLE` limits a *development* run; the full dataset should be used only after feature and split decisions are finalized.

**Do not present sample CV numbers as final results.** Change `SAMPLE = None` only after the team has agreed on the feature and evaluation protocol.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from URLSentra's root or notebooks/ folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.model_selection import train_test_split

from src import config, data, features, evaluate
from src.models import logreg, tree, forest, svm

SAMPLE = 3000  # entire cleaned data: None; smaller quick development run: 1500
SVM_TUNE_TRAIN_ROWS = 2000
RUN_FINAL_TEST = False  # IMPORTANT: keep False for ordinary Run All
FINAL_PROTOCOL_APPROVED = False  # final test requires the group's explicit approval
LEAKING_FEATURES = {"URLSimilarityIndex"}  # Group decision, 2026-11-19

print("Project root:", ROOT)
print("SAMPLE:", SAMPLE, "• RUN_FINAL_TEST:", RUN_FINAL_TEST)

## 2. Load/clean data and prepare common training split
Preprocessing stays in Member 1's `src/data.py`. The two feature sets use **exactly the same training rows**. Any feature selection that learns from data receives training rows only. We do not edit `src/features.py` here.

In [ ]:
if not config.RAW_FILE.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {config.RAW_FILE}. Download it from the UCI link above "
        "and place the CSV under data/raw/. Do not commit the dataset."
    )
df = data.load_clean_data()
if SAMPLE is not None and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)

original_cols = [
    c for c in data.get_original_features(df)
    if c not in LEAKING_FEATURES
]
df = features.add_features(df)
candidates = list(dict.fromkeys(original_cols + features.ENGINEERED_FEATURES))
X_train_all, _X_unused, y_train, _y_unused = data.split(df, candidates)
# Discard test references; DO NOT evaluate the test set during normal Run All.
del _X_unused, _y_unused
selected = features.select_features(df.loc[X_train_all.index], candidates)
engineered_cols = [c for c in selected if c not in LEAKING_FEATURES]
if not original_cols or not engineered_cols:
    raise ValueError("No features left after selecting features")

feature_sets = {
    "original": (X_train_all[original_cols], y_train),
    "engineered": (X_train_all[engineered_cols], y_train),
}
display(pd.DataFrame({
    "feature_set": list(feature_sets),
    "n_train_rows": [len(x) for x, _ in feature_sets.values()],
    "n_features": [x.shape[1] for x, _ in feature_sets.values()],
}))
print("Phishing positive-class ratio in training data:", round(float(y_train.mean()), 4))
print("Engineered feature list is provisional until Member 3 merges features.py")

## 3. SVM methodology and hyperparameter tuning (training only)
We use a **linear support vector classifier** (`LinearSVC`) because a kernel SVM may be expensive at ~235k rows. The `StandardScaler` is **inside** the calibrated classifier so scaling and probability calibration are refitted inside each cross-validation fold. The calibrated estimator supports `predict_proba()` needed for ROC-AUC and phishing risk scores.

Tune the regularization parameter **C** over `[0.01, 0.1, 1, 10]` with the team's `cv_splitter()` and phishing-class F1 as the selection metric. A subset of the *training split only* is used to control runtime. The `BEST_C` in `svm.py` remains a safe baseline until full, approved tuning is rerun and that value is deliberately updated.

In [ ]:
X_original, y_original = feature_sets["original"]
if SVM_TUNE_TRAIN_ROWS < len(X_original):
    indices, _ = train_test_split(
        X_original.index, train_size=SVM_TUNE_TRAIN_ROWS,
        stratify=y_original, random_state=config.RANDOM_STATE,
    )
    X_tune = X_original.loc[indices]
    y_tune = y_original.loc[indices]
else:
    X_tune, y_tune = X_original, y_original

print(f"Tuning SVM on {len(X_tune):,} TRAINING rows (no test rows) ...")
search = svm.tune(X_tune, y_tune, n_jobs=1)
cv_tuning = svm.tuning_table(search)
display(cv_tuning.round(5))
best_c = float(search.best_params_["estimator__linearsvc__C"])
print("Best C on this TRAINING CV sample:", best_c)
print("Mean CV F1 for selected C:", round(float(search.best_score_), 5))
print("Note: chosen C is provisional until full training and features are frozen.")

## 4. Shared model comparison: original vs engineered (TRAINING CV)
All four models are compared with **the same five-fold stratified training cross-validation**, on the same rows. We use configured estimators from each member's module. The SVM uses the C selected from the training-only tuning above. No test-set scores are used here. For a development run, this step uses only `SAMPLE` rows; the 500-tree forest may still be the slowest component.

In [ ]:
models = {
    "Logistic Regression": logreg.build(),
    "Decision Tree": tree.build(),
    "Random Forest": forest.build(),
    "SVM": svm.configured(C=best_c),
}
cv_results = evaluate.compare_training_models(feature_sets, models)
columns = ["feature_set", "model"] + [f"{m}_mean" for m in evaluate.CV_METRICS]
display(cv_results[columns].round(5))
# Keep full std columns for reproducibility and confidence/variability reporting.
display(cv_results[["feature_set", "model", "f1_mean", "f1_std", "roc_auc_mean", "roc_auc_std"]].round(5))

## 5. Model comparison chart and original-vs-engineered deltas
The plotted bars show **CV F1 means**, not held-out final scores. Absolute differences can be misleading near 1.0; always inspect class balance and leakage, not just overall accuracy.

In [ ]:
deltas = evaluate.feature_delta_table(cv_results)
display(deltas[["model", "f1_mean_original", "f1_mean_engineered", "delta_f1_mean"]].round(6))
fig1 = evaluate.plot_feature_comparison(
    cv_results, metric="f1", filename="member4_train_cv_feature_comparison.png"
)
plt.show()

## 6. ROC and confusion matrices — INTERNAL validation only
To visualize models without looking at the true final test data, split the already-held **training rows** into an inner fit (80%) and inner validation (20%) portion. Every ROC curve and confusion matrix uses this identical inner validation partition; it is **diagnostic**, not an official final estimate. The five-fold CV table above is the main training-only comparison.

In [ ]:
X_inner, X_valid, y_inner, y_valid = train_test_split(
    X_original, y_original,
    test_size=config.TEST_SIZE, stratify=y_original,
    random_state=config.RANDOM_STATE,
)
fitted_inner = {}
for name, prototype in models.items():
    print("Inner validation fit:", name)
    fitted_inner[name] = clone(prototype).fit(X_inner, y_inner)

fig2 = evaluate.plot_roc_curves(
    fitted_inner, X_valid, y_valid,
    title="URLSentra: Training-only inner-validation ROC",
    filename="member4_train_validation_roc.png",
)
plt.show()
fig3 = evaluate.plot_confusion_matrices(
    fitted_inner, X_valid, y_valid,
    title="Training-only inner-validation confusion matrices",
    filename="member4_train_validation_confusion.png",
)
plt.show()

## 7. Results and interpretation (automatically derived from the actual run)
This section displays the observed numbers from **the actual data loaded above**. It does not invent numerical results. Record the SVM's best C and the difference between original and engineered features after executing this notebook.

**Interpretation rules:** Positive label is phishing. Recall tells us how many phishing URLs are detected; precision tells us what fraction of phishing alerts are correct; F1 balances them; ROC-AUC measures ranking across decision thresholds. Compare the uncertainty/variation across folds. A training sample is not a substitute for an independent external-domain validation.

In [ ]:
original_cv = cv_results.loc[cv_results.feature_set == "original"].copy()
best_row = original_cv.loc[original_cv.f1_mean.idxmax()]
svm_original = original_cv.loc[original_cv.model == "SVM"].iloc[0]
svm_deltas = deltas.loc[deltas.model == "SVM"].iloc[0]

print("OBSERVED TRAINING-CV SUMMARY (not final test results)")
print(f"Best original-feature mean CV F1: {best_row['model']} = {best_row['f1_mean']:.5f}")
print(f"SVM original-feature CV F1: {svm_original.f1_mean:.5f} ± {svm_original.f1_std:.5f}")
print(f"SVM engineered-minus-original CV F1: {svm_deltas['delta_f1_mean']:+.5f}")
if (cv_results.f1_mean >= 0.99).any():
    display(Markdown(
        "**WARNING — near-perfect F1 observed.** This should trigger leakage and "
        "sampling-bias investigation, **not** a claim of solved phishing detection. "
        "`URLSimilarityIndex` was removed by group decision, but `IsHTTPS`, "
        "webpage/content features, near-duplicates and random rather than "
        "domain-grouped splitting remain important concerns."
    ))
display(Markdown(
    "**Limitation:** These are sampled training CV results with Member 3's "
    "current feature interface. Re-run after feature engineering/selection is merged "
    "and after group decisions on near-duplicates, domains and biased page features. "
    "No held-out final-test metric has been calculated during this notebook run."
))

## 8. Optional FINAL held-out test (disabled by default)
**Do not enable** until (1) Member 3's actual feature engineering has merged, (2) the group has approved the split/leakage protocol, and (3) model selection is frozen. Running the cell in ordinary **Run All** produces **no test predictions**. If deliberately enabled after approval, the chosen SVM is fitted on the full training rows and evaluated **once** on the held-out split.

In [ ]:
if RUN_FINAL_TEST:
    if SAMPLE is not None:
        raise RuntimeError("Final evaluation requires SAMPLE = None (the full cleaned dataset).")
    if not FINAL_PROTOCOL_APPROVED:
        raise RuntimeError(
            "Final evaluation blocked: obtain group approval, then set "
            "FINAL_PROTOCOL_APPROVED = True."
        )
    X_train_final, X_test_final, y_train_final, y_test_final = data.split(df, original_cols)
    final_svm = svm.configured(C=best_c).fit(X_train_final, y_train_final)
    final_scores = evaluate.evaluate(final_svm, X_test_final, y_test_final)
    display(pd.DataFrame([{"model": "SVM", "feature_set": "original", **final_scores}]))
else:
    print("Final held-out test SKIPPED (RUN_FINAL_TEST = False).")

## 9. Experiment log and handover notes
| Step | Why | Evidence on execution | Status |
| --- | --- | --- | --- |
| Exclude `URLSimilarityIndex` | Group-confirmed label leakage | Feature names in section 2 | Applied |
| LinearSVC + calibration | Fast linear baseline and probability estimates | `svm.configured()` implementation | Implemented |
| Tune C | Choose model complexity using training CV | Section 3 CV search table |  Completed - C=0.1, CV F1=0.99715 (provisional)|
| Compare four models, two feature sets | Fair common CV metrics | Section 4 table, section 5 chart | Provisional until Member 3 merge |
| Validation ROC and confusion matrices | Visualize false positives and missed phishing | Section 6 figures | Training-only diagnostics |
| Official full-data and test reporting | Unbiased reported final result | Section 8 guard | **Not performed** |

**Required rerun after Member 3 merges `src/features.py`:** sections 2–7, with the agreed clean/split protocol. Then re-tune SVM on an approved larger training sample, update `BEST_C` in `svm.py` only with the verified optimum, regenerate all figures and comparison tables, and finally run the guarded test **once** with group consent.

**Known limitations:** near-duplicate URLs, potentially overlapping domains across random train/test partitions, `IsHTTPS`/page-feature collection bias, sample-based tuning, unfinalized risk bands and possible high model scores caused by hidden leakage. No official final accuracy/ROC-AUC is claimed here.

**Suggested verification commands (repo root):**
```bash
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
jupyter notebook notebooks/03_evaluation_svm.ipynb
python main.py --sample 20000 --explain-rows 0
```
When the group is ready, use `python main.py` on the full dataset. Do **not** commit CSV datasets, fitted models or notebook outputs.